# Circuitos electorales de Argentina

Recorre los GeoJSON de `2021/` y `2025/`, verifica que el esquema sea uniforme y
genera las tablas que publica el `README.md`.

Solo necesita `pandas`: el conteo se hace leyendo las propiedades, sin cargar
geometrías, así que corre en segundos aunque el repositorio pese ~110 MB.

In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
ANIOS = ["2021", "2025"]

PROVINCIAS = {
    "01": "Ciudad Autónoma de Buenos Aires", "02": "Buenos Aires",
    "03": "Catamarca", "04": "Córdoba", "05": "Corrientes",
    "06": "Chaco", "07": "Chubut", "08": "Entre Ríos",
    "09": "Formosa", "10": "Jujuy", "11": "La Pampa",
    "12": "La Rioja", "13": "Mendoza", "14": "Misiones",
    "15": "Neuquén", "16": "Río Negro", "17": "Salta",
    "18": "San Juan", "19": "San Luis", "20": "Santa Cruz",
    "21": "Santa Fe", "22": "Santiago del Estero", "23": "Tucumán",
    "24": "Tierra del Fuego",
}

def es_codigo(circuito):
    """Distingue un código de circuito de los marcadores sin dato.

    Córdoba trae polígonos con 'sindatos' o 'zonagris', y Formosa y Santa Cruz
    2025 traen 'S/D'. Cubren territorio sin circuito asignado en la fuente.
    """
    return isinstance(circuito, str) and circuito[:1].isdigit()

In [ ]:
def leer_anio(anio):
    """Devuelve un DataFrame con una fila por feature del año indicado."""
    filas = []
    for ruta in sorted(Path(anio).glob("*.geojson")):
        gj = json.loads(ruta.read_text(encoding="utf-8"))
        for feature in gj["features"]:
            p = feature["properties"]
            filas.append({
                "anio": anio,
                "archivo": ruta.stem,
                "codprov": p["codprov"],
                "coddepto": p["coddepto"],
                "circuito": p["circuito"],
                "tipo_geometria": (feature["geometry"] or {}).get("type"),
            })
    df = pd.DataFrame(filas)
    df["provincia"] = df.codprov.map(PROVINCIAS)
    df["es_codigo"] = df.circuito.map(es_codigo)
    return df


df = pd.concat([leer_anio(a) for a in ANIOS], ignore_index=True)
print(f"{len(df)} features leídos de {df.archivo.nunique()} distritos × {len(ANIOS)} años")
df.head()

## 1. Chequeos de consistencia

Antes de contar, confirmar que el dato es el que se espera.

In [ ]:
problemas = []

# Un archivo por distrito en cada año
for anio in ANIOS:
    n = df[df.anio == anio].archivo.nunique()
    if n != 24:
        problemas.append(f"{anio}: {n} distritos en vez de 24")

# codprov constante dentro de cada archivo y presente en la tabla de provincias
for (anio, archivo), g in df.groupby(["anio", "archivo"]):
    if g.codprov.nunique() != 1:
        problemas.append(f"{anio}/{archivo}: codprov no es constante: {sorted(g.codprov.unique())}")
    if g.provincia.isna().any():
        problemas.append(f"{anio}/{archivo}: codprov desconocido {sorted(g.codprov.unique())}")

# Longitud del código de circuito
cortos = df[df.es_codigo & (df.circuito.str.len() < 5)]
if len(cortos):
    problemas.append(f"{len(cortos)} circuitos con menos de 5 caracteres")

print("\n".join(problemas) if problemas else "Sin problemas de consistencia.")

In [ ]:
# Limitaciones conocidas del dato de origen: se reportan, no se corrigen.
nulos = df[df.coddepto.isna()]
if len(nulos):
    print(f"coddepto nulo en {len(nulos)} features:")
    print(nulos.groupby(["anio", "archivo"]).size().to_string())

sin_codigo = df[~df.es_codigo]
if len(sin_codigo):
    print(f"\nFeatures sin código de circuito: {len(sin_codigo)}")
    print(sin_codigo.groupby(["anio", "archivo"]).circuito
          .agg(lambda s: ", ".join(sorted(set(s)))).to_string())

## 2. Circuitos por distrito y año

In [ ]:
validos = df[df.es_codigo]

resumen = (
    validos.groupby(["anio", "codprov", "provincia", "archivo"])
    .agg(circuitos=("circuito", "nunique"), features=("circuito", "size"))
    .reset_index()
)

tabla = resumen.pivot_table(
    index=["codprov", "provincia", "archivo"],
    columns="anio",
    values="circuitos",
    aggfunc="sum",
).astype(int).reset_index()
tabla.columns.name = None

tabla["delta"] = tabla["2025"] - tabla["2021"]
tabla = tabla.sort_values("codprov")
display(tabla)

print(f"Total 2021: {tabla['2021'].sum()}   "
      f"Total 2025: {tabla['2025'].sum()}   "
      f"Δ: {tabla['delta'].sum():+d}")

## 3. Comparabilidad entre años

Cuántos códigos de 2021 siguen existiendo en 2025. Un valor bajo indica que la
fuente cambió de convención o que la provincia fue redibujada, y que los dos
cortes no se pueden unir directamente en ese distrito.

In [ ]:
comparacion = []
for archivo, g in validos.groupby("archivo"):
    a = set(g[g.anio == "2021"].circuito)
    b = set(g[g.anio == "2025"].circuito)
    comparacion.append({
        "archivo": archivo,
        "2021": len(a),
        "2025": len(b),
        "comunes": len(a & b),
        "solo_2021": len(a - b),
        "solo_2025": len(b - a),
        "pct_comun": round(100 * len(a & b) / len(a), 1) if a else None,
    })

comparacion = pd.DataFrame(comparacion).sort_values("pct_comun")
display(comparacion)

## 4. Exportar a CSV (opcional)

Las salidas están en `.gitignore`: son derivadas, se regeneran con este notebook
y no se versionan.

In [ ]:
# validos.to_csv("circuitos_detalle.csv", index=False)
# tabla.to_csv("circuitos_resumen.csv", index=False)
# print("Exportado.")